### Import dependencies

In [1]:
import openai
import os
import json
from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue
from langsmith import Client

In [2]:
from dotenv import load_dotenv
import os

load_dotenv("../../.env")

True

In [3]:
qdrant_client = QdrantClient(url='http://localhost:6333')

In [5]:
all_points = qdrant_client.scroll(
    collection_name='amazon-items-collection-00',
    limit=100,
    offset=None,
    with_payload=True,
    with_vectors=False,
)

In [6]:
all_context = [{'id':data.payload['parent_asin'], 'text':data.payload['processed_description']} for data in all_points[0]]

In [7]:
all_context 


[{'id': 'B0BRJS644Z',
  'text': 'Marame 120mm 5v USB Powered Fan with Speed Controller Cooling for Router Modem Receiver DVR Xbox TV Box (120mm x 120mm x 55mm) 【Solve Your Cooling Problem】\xa0This fan will do the job keeping your devices from overheating and keep your electronics running cool. You can also use them in confined spaces for cooling various electronics. blowing cool air through it to aid in longevity. 【Speed\xa0Control\xa0Switch】 The speed controller located on the cord allows you to adjust the fan’s speed from off to low, medium, and high. This enables you to set the fan to optimal noise and airflow levels for various environments. 【High Compatibility USB-Powered】 Powered by a 3.3ft USB cable. Compatible with desktop, laptop, power bank, AC adapters, car chargers, and other power supplies that support USB connection. USB fan is energy-saving and environmentally friendly. 【Rubber Feet & Dust Filter】\xa0Rubber Feet on the fan raise it up enough off of the surface it is sitt

In [10]:
import json
import os
from google import genai
from google.genai import types
from dotenv import load_dotenv

load_dotenv()

gemini_client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

output_schema = {
    'type': 'array',
    'items': {
        'type': 'object',
        'properties': {
            'reasoning': {
                'type': 'string',
                'description': 'Reasoning why the question could be answered with the given chunks.',
            },
            'question': {
                'type': 'string',
                'description': 'Suggested question.',
            },
            'chunk_ids': {
                'type': 'array',
                'items': {
                    'type': 'string',
                    'description': 'ID of the chunk that could be used to answer the question.',
                },
            },
            'answer_example': {
                'type': 'string',
                'description': 'Suggested answer grounded in the context.',
            },
        },
    },
}

SYSTEM_PROMPT = f"""
I am building a RAG application. I have a collection of 50 chunks of text.
The RAG application will act as shopping assistant that can answer questions about the stock of the products we have available.
I will provide all of the available products to you with IDs of each chunk
Come up with 30 questions to which the answers could be grounded in the chunk context.
The questions should imitate a potential real user of this RAG system - a customer of the e-shop.
As an output I need you to provide me the list of questions and the IDs of the chunks that could be used to answer them.
Also provide an example answer to the question given the context of the chunks
Also provide the reason why you chose the chunks to answer the question.
Construct 10 questions that could use multiple chunks in the answer.
Construct 15 questions that could use single chunk in the answer.
Construct 5 questions that could use no chunks at all.
Do not use word chunks in suggested questions, refer to the chunks as products.

<OUTPUT_JSON_SCHEMA>
{json.dumps(output_schema, indent=2)}
</OUTPUT_JSON_SCHEMA>

I need to be able to parse the json output.
"""

USER_PROMPT = f"""
Here is the list of chunks, each list element is a dictionary with id and text:
{json.dumps(all_context, indent=2)}
"""

response = gemini_client.models.generate_content(
    model="gemini-3.6-flash",
    contents=USER_PROMPT,
    config=types.GenerateContentConfig(
        system_instruction=SYSTEM_PROMPT,
        response_mime_type="application/json",
    ),
)

questions = json.loads(response.text)
print(f"Generated {len(questions)} questions")
print(json.dumps(questions[0], indent=2))

Generated 30 questions
{
  "reasoning": "Both products are designed to hold or cool laptops, allowing comparison between an active laptop fan pad and a basic desk riser.",
  "question": "What options do you have if I'm looking for a laptop stand or cooling pad for a 15.6 inch laptop?",
  "chunk_ids": [
    "B099N9F3FP",
    "B09YHG2Z7F"
  ],
  "answer_example": "We have two great products: 1) The AICHESON Laptop Cooler Pad (B099N9F3FP), which features 6 cooling fans, adjustable ergonomic heights (10 to 27 degrees), low noise, and fits laptops up to 15.6 inches. 2) The Simple Trending 2 Pack Monitor Stand Riser (B09YHG2Z7F), a metal desk organizer stand that elevates your laptop or monitor up to eye level and holds up to 50 pounds."
}


In [12]:
with open("synthetic_questions.json", "w") as f:
    json.dump(questions, f, indent=2)

print("Saved to synthetic_questions.json")

Saved to synthetic_questions.json


In [13]:
multi  = [q for q in questions if len(q["chunk_ids"]) > 1]
single = [q for q in questions if len(q["chunk_ids"]) == 1]
none   = [q for q in questions if len(q["chunk_ids"]) == 0]

print(f"Multi-chunk:  {len(multi)}")   # target ~10
print(f"Single-chunk: {len(single)}")  # target ~15
print(f"No chunks:    {len(none)}")    # target ~5

Multi-chunk:  10
Single-chunk: 15
No chunks:    5


In [16]:
response = gemini_client.models.generate_content(
    model="gemini-3.6-flash",
    contents=USER_PROMPT,
    config=types.GenerateContentConfig(
        system_instruction=SYSTEM_PROMPT,
        response_mime_type="application/json",
    ),
)
json_output = json.loads(response.text)

In [17]:
json_output = json.loads(response.text)

In [18]:
len(json_output)

30

### Fetch the item descriptions based on the chunk IDs

In [24]:
def get_descriptions(parent_asin: str) -> str:
    points = qdrant_client.scroll(
        collection_name='amazon-items-collection-01',
        scroll_filter=Filter(
            must=[
                FieldCondition(key='parent_asin', match=MatchValue(value=parent_asin))
            ]
        ),
    )[0]

    return points[0].payload['processed_description']

### Create Eval dataset in Langsmith

In [25]:
ls_client = Client(api_key=os.environ["LANGSMITH_API_KEY"])
dataset_name = 'rag-eval-dataset'
dataset = ls_client.create_dataset(
    dataset_name=dataset_name,
    description='Dataset for Evaluating Rag pipeline',
)

LangSmithConflictError: Conflict for /datasets. HTTPError('409 Client Error: Conflict for url: https://api.smith.langchain.com/datasets', '{"detail":"Dataset with this name already exists."}')

In [26]:
for item in json_output:
    Client.create_example(
        dataset_id=dataset.id,
        inputs={"question": item["question"]},
        outputs={
            "ground_truth": item["answer_example"],
            "reference_context_ids": item["chunk_ids"],
            "reference_descriptions": [get_description(id) for id in item["chunk_ids"]]
        }
    )

NameError: name 'get_description' is not defined